# 01 — Install dependencies and prepare the source data
Run this notebook first. Install the libraries once in the Jupyter environment; restart the kernel after an installation if imports require it. The source CSV stays unchanged. The two buckets and batch objects are created here, when you run these cells.

In [1]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 29.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 121.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 116.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 48.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 48.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.9/785.9 kB 29.6 MB/s  0:00:00
  Attempting uninstall: wcwidth━━━━━━━━━━━━━━━━━━━━━━━━━━  1/46 [werkzeug]
    Found existing installation: wcwidth 0.2.13━━━━━━  1/46 [werkzeug]
    Uninstalling wcwidth-0.2.13:━━━━━━━━━━━━━━━━━━━━━  1/46 [werkzeug]
      Successfully uninstalled wcwidth-0.2.13━━━━━━━━  1/46 [werkzeug]
  Attempting uninstall: typing-extensions━━━━━━━━━━━━━━━━━━━━━━━━━  3/46 [uvicorn]
   

In [2]:
import sys
from pathlib import Path
project = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from botocore.exceptions import ClientError

## Functions: settings
This is a copy; edit the source in 05_pipeline_walkthrough.ipynb. Run the cells in order. After saving edits in the source notebooks, run the synchronization command described in the README.

In [3]:
import os
from urllib.parse import quote_plus


In [4]:
def s3_client():
    import boto3
    return boto3.client('s3', endpoint_url=os.getenv('S3_ENDPOINT_URL','http://minio.mlops-project.svc.cluster.local:9000'), region_name='us-east-1', aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'], aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'])


## Inspect the original CSV
The prediction target used in this project is **readmission within 30 days**, recorded as `<30`. Each row is an encounter; a patient can have several encounters.

In [5]:
source = project / 'data' / 'diabetic_data.csv'
if not source.exists():
    raise FileNotFoundError(f'Source CSV not found: {source}')
data = pd.read_csv(source, dtype=str, keep_default_na=False)
required = {'encounter_id', 'patient_nbr', 'readmitted'}
assert required.issubset(data.columns) and data.encounter_id.is_unique
print(f'{len(data):,} encounters; {data.patient_nbr.nunique():,} patients; {len(data.columns)} columns')
display(data.readmitted.value_counts().rename_axis('readmission').to_frame('encounters'))
display(data.head(3))

101,766 encounters; 71,518 patients; 50 columns


,encounters
readmission,
NO,54864
>30,35545
<30,11357


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## Create the buckets
`diabetes-source` holds source batches. `mlflow-artifacts` holds figures and models logged by MLflow.

In [6]:
s3 = s3_client()
for bucket in ['diabetes-source', 'mlflow-artifacts']:
    try:
        s3.head_bucket(Bucket=bucket)
        print('Already exists:', bucket)
    except ClientError as error:
        if error.response['Error']['Code'] in ('404', 'NoSuchBucket', 'NotFound'):
            s3.create_bucket(Bucket=bucket)
            print('Created:', bucket)
        else:
            raise

Created: diabetes-source
Created: mlflow-artifacts


## Upload the batches
Each object contains at most 15,000 rows in source order. Running the cell again replaces objects with the same names, so keep the source file unchanged after processing begins.

In [7]:
batch_size = 15_000
summary = []
for number, start in enumerate(range(0, len(data), batch_size), 1):
    batch = data.iloc[start:start + batch_size]
    key = f'batches/batch_{number:04d}.csv'
    s3.put_object(Bucket='diabetes-source', Key=key,
                  Body=batch.to_csv(index=False).encode('utf-8'), ContentType='text/csv')
    summary.append({'object': key, 'rows': len(batch), 'first_source_row': start + 1})
display(pd.DataFrame(summary))
print('Total uploaded rows:', sum(item['rows'] for item in summary))

,object,rows,first_source_row
0,batches/batch_0001.csv,15000,1
1,batches/batch_0002.csv,15000,15001
2,batches/batch_0003.csv,15000,30001
3,batches/batch_0004.csv,15000,45001
4,batches/batch_0005.csv,15000,60001
5,batches/batch_0006.csv,15000,75001
6,batches/batch_0007.csv,11766,90001


Total uploaded rows: 101766


In [8]:
objects = s3.list_objects_v2(Bucket='diabetes-source', Prefix='batches/').get('Contents', [])
display(pd.DataFrame([{'object': x['Key'], 'bytes': x['Size']} for x in objects]))

,object,bytes
0,batches/batch_0001.csv,2813936
1,batches/batch_0002.csv,2823020
2,batches/batch_0003.csv,2878196
3,batches/batch_0004.csv,2812455
4,batches/batch_0005.csv,2796295
5,batches/batch_0006.csv,2769134
6,batches/batch_0007.csv,2168582
